# TFT Match History Data Pipeline
Fetches match data from the Riot Games API and stores it in a local SQLite database.

**Setup:**
1. Copy `.env.example` → `.env` and add your Riot API key
2. Run `pip install -r requirements.txt`
3. Run all cells in order

In [ ]:
import os
import time
import json
import requests
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load API key and config from .env (never hard-code secrets!)
load_dotenv()

API_KEY = os.getenv("RIOT_API_KEY")
DB_PATH = os.getenv("DB_PATH", "tft_data.db")

if not API_KEY:
    raise EnvironmentError("RIOT_API_KEY not found. Did you create a .env file?")

print("✅ API key loaded")
print(f"📦 Database path: {DB_PATH}")

✅ API key loaded
📦 Database path: tft_data.db


## 1. Create the SQLite Database Schema

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}")

SCHEMA_SQL = """
-- Summoner/Player information
CREATE TABLE IF NOT EXISTS summoners (
    puuid          TEXT PRIMARY KEY,
    game_name      TEXT,
    tag_line       TEXT,
    summoner_id    TEXT,
    region         TEXT,
    fetched_at     TEXT DEFAULT (datetime('now'))
);

-- One row per match
CREATE TABLE IF NOT EXISTS matches (
    match_id       TEXT PRIMARY KEY,
    game_datetime  INTEGER,   -- Unix ms timestamp
    game_length    REAL,      -- seconds
    game_variation TEXT,
    tft_set_number INTEGER,
    tft_set_core_name TEXT,
    queue_id       INTEGER,
    fetched_at     TEXT DEFAULT (datetime('now'))
);

-- One row per player per match (8 players per TFT game)
CREATE TABLE IF NOT EXISTS match_participants (
    id                   INTEGER PRIMARY KEY AUTOINCREMENT,
    match_id             TEXT REFERENCES matches(match_id),
    puuid                TEXT REFERENCES summoners(puuid),
    placement            INTEGER,
    level                INTEGER,
    last_round           INTEGER,
    players_eliminated   INTEGER,
    total_damage_to_players REAL,
    gold_left            INTEGER,
    time_eliminated      REAL,
    augments             TEXT,  -- JSON array
    UNIQUE(match_id, puuid)
);

-- Units fielded by each participant
CREATE TABLE IF NOT EXISTS participant_units (
    id               INTEGER PRIMARY KEY AUTOINCREMENT,
    match_id         TEXT,
    puuid            TEXT,
    character_id     TEXT,
    tier             INTEGER,
    rarity           INTEGER,
    items            TEXT,   -- JSON array of item IDs
    chosen_trait     TEXT,
    FOREIGN KEY (match_id, puuid) REFERENCES match_participants(match_id, puuid)
);

-- Traits active for each participant
CREATE TABLE IF NOT EXISTS participant_traits (
    id               INTEGER PRIMARY KEY AUTOINCREMENT,
    match_id         TEXT,
    puuid            TEXT,
    trait_name       TEXT,
    num_units        INTEGER,
    style            INTEGER,  -- 0=None,1=Bronze,2=Silver,3=Gold,4=Chromatic
    tier_current     INTEGER,
    tier_total       INTEGER,
    FOREIGN KEY (match_id, puuid) REFERENCES match_participants(match_id, puuid)
);
"""

with engine.connect() as conn:
    for statement in SCHEMA_SQL.strip().split(";"):
        stmt = statement.strip()
        if stmt:
            conn.execute(text(stmt))
    conn.commit()

print("✅ Database schema created:", DB_PATH)

✅ Database schema created: tft_data.db


: 

: 

## 2. Riot API Helper Functions

In [ ]:
# --- Configuration ---
# Routing values: americas | europe | asia | sea
REGIONAL_ROUTING = "sea"        # OCE players route via sea
# Platform routing: oc1, na1, euw1, kr, etc.
PLATFORM_ROUTING = "oc1"

HEADERS = {"X-Riot-Token": API_KEY}
BASE_REGIONAL = f"https://{REGIONAL_ROUTING}.api.riotgames.com"
BASE_PLATFORM  = f"https://{PLATFORM_ROUTING}.api.riotgames.com"

def _get(url, params=None, retries=3):
    """GET with basic rate-limit handling (HTTP 429 back-off)."""
    for attempt in range(retries):
        r = requests.get(url, headers=HEADERS, params=params)
        if r.status_code == 200:
            return r.json()
        elif r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 5))
            print(f"  Rate limited — waiting {wait}s...")
            time.sleep(wait)
        elif r.status_code == 404:
            print(f"  404 Not Found: {url}")
            return None
        else:
            print(f"  HTTP {r.status_code}: {url}")
            return None
    return None


def get_account_by_riot_id(game_name: str, tag_line: str) -> dict:
    """Resolve Riot ID (name#tag) to PUUID via Account API."""
    url = f"{BASE_REGIONAL}/riot/account/v1/accounts/by-riot-id/{game_name}/{tag_line}"
    return _get(url)


def get_match_ids(puuid: str, count: int = 20, start: int = 0) -> list:
    """Fetch a list of TFT match IDs for a PUUID."""
    url = f"{BASE_REGIONAL}/tft/match/v1/matches/by-puuid/{puuid}/ids"
    return _get(url, params={"count": count, "start": start}) or []


def get_match_detail(match_id: str) -> dict:
    """Fetch full match detail for a single match ID."""
    url = f"{BASE_REGIONAL}/tft/match/v1/matches/{match_id}"
    return _get(url)


print("✅ API helpers ready")
print(f"   Regional routing : {REGIONAL_ROUTING}")
print(f"   Platform routing : {PLATFORM_ROUTING}")

✅ API helpers ready
   Regional routing : sea
   Platform routing : oc1


: 

: 

## 3. Database Insert Helpers

In [ ]:
def upsert_summoner(conn, puuid, game_name, tag_line, region):
    conn.execute(text("""
        INSERT INTO summoners (puuid, game_name, tag_line, region)
        VALUES (:puuid, :game_name, :tag_line, :region)
        ON CONFLICT(puuid) DO UPDATE SET
            game_name = excluded.game_name,
            tag_line  = excluded.tag_line
    """), {"puuid": puuid, "game_name": game_name, "tag_line": tag_line, "region": region})


def match_already_stored(conn, match_id):
    row = conn.execute(text("SELECT 1 FROM matches WHERE match_id = :m"), {"m": match_id}).fetchone()
    return row is not None


def insert_match(conn, match_id, info):
    conn.execute(text("""
        INSERT OR IGNORE INTO matches
            (match_id, game_datetime, game_length, game_variation,
             tft_set_number, tft_set_core_name, queue_id)
        VALUES
            (:match_id, :game_datetime, :game_length, :game_variation,
             :tft_set_number, :tft_set_core_name, :queue_id)
    """), {
        "match_id"        : match_id,
        "game_datetime"   : info.get("game_datetime"),
        "game_length"     : info.get("game_length"),
        "game_variation"  : info.get("game_variation", ""),
        "tft_set_number"  : info.get("tft_set_number"),
        "tft_set_core_name": info.get("tft_set_core_name", ""),
        "queue_id"        : info.get("queue_id"),
    })


def insert_participant(conn, match_id, p):
    conn.execute(text("""
        INSERT OR IGNORE INTO match_participants
            (match_id, puuid, placement, level, last_round,
             players_eliminated, total_damage_to_players, gold_left,
             time_eliminated, augments)
        VALUES
            (:match_id, :puuid, :placement, :level, :last_round,
             :players_eliminated, :total_damage_to_players, :gold_left,
             :time_eliminated, :augments)
    """), {
        "match_id"                : match_id,
        "puuid"                   : p["puuid"],
        "placement"               : p.get("placement"),
        "level"                   : p.get("level"),
        "last_round"              : p.get("last_round"),
        "players_eliminated"      : p.get("players_eliminated"),
        "total_damage_to_players" : p.get("total_damage_to_players"),
        "gold_left"               : p.get("gold_left"),
        "time_eliminated"         : p.get("time_eliminated"),
        "augments"                : json.dumps(p.get("augments", [])),
    })


def insert_units(conn, match_id, puuid, units):
    for u in units:
        conn.execute(text("""
            INSERT INTO participant_units
                (match_id, puuid, character_id, tier, rarity, items, chosen_trait)
            VALUES
                (:match_id, :puuid, :character_id, :tier, :rarity, :items, :chosen_trait)
        """), {
            "match_id"     : match_id,
            "puuid"        : puuid,
            "character_id" : u.get("character_id"),
            "tier"         : u.get("tier"),
            "rarity"       : u.get("rarity"),
            "items"        : json.dumps(u.get("itemNames", u.get("items", []))),
            "chosen_trait" : u.get("chosen", ""),
        })


def insert_traits(conn, match_id, puuid, traits):
    for t in traits:
        conn.execute(text("""
            INSERT INTO participant_traits
                (match_id, puuid, trait_name, num_units, style, tier_current, tier_total)
            VALUES
                (:match_id, :puuid, :trait_name, :num_units, :style, :tier_current, :tier_total)
        """), {
            "match_id"    : match_id,
            "puuid"       : puuid,
            "trait_name"  : t.get("name"),
            "num_units"   : t.get("num_units"),
            "style"       : t.get("style"),
            "tier_current": t.get("tier_current"),
            "tier_total"  : t.get("tier_total"),
        })


print("✅ DB insert helpers ready")

✅ DB insert helpers ready


: 

: 

## 4. Fetch & Store Match History
Edit `PLAYERS` with the Riot IDs you want to track.

In [ ]:
# --- Edit these ---
PLAYERS = [
    {"game_name": "nichbane2369", "tag_line": "OCE"},
    # Add more players here
]
MATCHES_PER_PLAYER = 100   # max 200 with personal API key; 200 with production key
DELAY_BETWEEN_REQUESTS = 1.2  # seconds — personal key: 20 req/s, 100 req/2min


def fetch_and_store_player(game_name: str, tag_line: str):
    print(f"\n👤 Processing: {game_name}#{tag_line}")

    # 1. Resolve Riot ID → PUUID
    account = get_account_by_riot_id(game_name, tag_line)
    if not account:
        print("  ❌ Could not resolve Riot ID")
        return
    puuid = account["puuid"]

    # 2. Save summoner
    with engine.begin() as conn:
        upsert_summoner(conn, puuid, game_name, tag_line, PLATFORM_ROUTING)

    # 3. Get match IDs
    time.sleep(DELAY_BETWEEN_REQUESTS)
    match_ids = get_match_ids(puuid, count=MATCHES_PER_PLAYER)
    print(f"  Found {len(match_ids)} matches")

    # 4. Fetch & store each match
    new_matches = 0
    for match_id in match_ids:
        with engine.begin() as conn:
            if match_already_stored(conn, match_id):
                continue  # skip duplicates

        time.sleep(DELAY_BETWEEN_REQUESTS)
        match = get_match_detail(match_id)
        if not match:
            continue

        info = match["info"]
        with engine.begin() as conn:
            insert_match(conn, match_id, info)
            for p in info.get("participants", []):
                insert_participant(conn, match_id, p)
                insert_units(conn, match_id, p["puuid"], p.get("units", []))
                insert_traits(conn, match_id, p["puuid"], p.get("traits", []))
        new_matches += 1

    print(f"  ✅ Stored {new_matches} new matches")


# Run for all players
for player in PLAYERS:
    fetch_and_store_player(player["game_name"], player["tag_line"])

print("\n🎉 Done!")


👤 Processing: nichbane2369#OC
  HTTP 403: https://sea.api.riotgames.com/riot/account/v1/accounts/by-riot-id/nichbane2369/OC
  ❌ Could not resolve Riot ID

🎉 Done!


: 

: 

In [ ]:
from dotenv import load_dotenv
load_dotenv()
key = os.getenv("RIOT_API_KEY")
print(repr(key)) 

'RGAPI-937204a8-78c4-444f-9c8c-1a4b5ed5059c'


: 

: 

## 5. Quick Sanity Check

In [ ]:
with engine.connect() as conn:
    for table in ["summoners", "matches", "match_participants", "participant_units", "participant_traits"]:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"  {table:<30} {count:>6} rows")

: 

: 

In [ ]:
# Example: average placement per augment
query = """
SELECT
    json_each.value       AS augment,
    ROUND(AVG(mp.placement), 2) AS avg_placement,
    COUNT(*)              AS games
FROM match_participants mp,
     json_each(mp.augments)
GROUP BY augment
HAVING games >= 5
ORDER BY avg_placement
LIMIT 20;
"""
df = pd.read_sql(query, engine)
df

: 

: 